# Día 19 — Limpieza y Joins en pandas

**Dataset:** Empresa de servicios — `empleados.csv` + `proyectos.csv`

---

In [ ]:
import pandas as pd

dfEmpleados = pd.read_csv('../data/day19/empleados.csv')
dfProyectos  = pd.read_csv('../data/day19/proyectos.csv')

print('Empleados:', dfEmpleados.shape)
print('Proyectos:', dfProyectos.shape)
print('--------------------------------')
print('Empleados:', dfEmpleados.dtypes)
print('________________________________')
print('Proyectos:', dfProyectos.dtypes)
print('--------------------------------')
print('Información estructurada sobre el dataset: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.info())
print('________________________________')
print('Proyectos:', dfProyectos.info())
print('--------------------------------')
print('Primeros registros y nombre de cada campo: ')
print('--------------------------------')
print('Empleados:', dfEmpleados.head())
print('________________________________')
print('Proyectos:', dfProyectos.head())
print('--------------------------------')

---
## 1 - Detección de nulos

Se cuantifican los nulos por columna en `empleados` y el porcentaje que representa cada uno sobre el total de registros.

In [ ]:
# La identificación de nulos se con isnull(), que devuelve True donde hay NaN, y luego .sum() cuenta esos True
nulos = dfEmpleados.isnull().sum()

#Para saber el porcentaje de nulos se dicide la cantidad de nulos acumulada por el total de registros
ptcNulos = (nulos/ len(dfEmpleados)*100).round(2) 

#Se crea una tabla de muestreo con campos filtrados/nuevos 
# con un diccionario, donde cada clave equivale a una columna
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje_%': ptcNulos})

# Se muestra el resumen filtrando unicamente por donde existen dentro de resumen
print(resumen[resumen['nulos']>0])

---
## 2 - Tratamiento de nulos

Los nulos en `departamento` se rellenan con `'Sin asignar'`. Las filas con `salario` o `fecha_ingreso` nulos se eliminan.

In [ ]:
# Departamento nulo equivale a rellenar campo nulo con valor asignado por fillna() = 'Sin asignar'
dfEmpleados['departamento'] = dfEmpleados['departamento'].fillna('Sin asignar')

filas_antes = len(dfEmpleados)

dfEmpleados = dfEmpleados.dropna(subset=['salario'])
dfEmpleados = dfEmpleados.dropna(subset=['fecha_ingreso'])

print('Filas eliminadas por salario o fecha nulos:', filas_antes - len(dfEmpleados))
print('Nulos restantes en salario, departamento y fecha de ingreso :')
print(dfEmpleados[['salario', 'departamento', 'fecha_ingreso']].isnull().sum())

---
## 3 - Duplicados

Se identifican y eliminan las filas duplicadas en `empleados`, mostrando qué registros estaban repetidos.

In [ ]:
print('Filas duplicadas: ',dfEmpleados.duplicated().sum())
print('Duplicados: ')
print(dfEmpleados[dfEmpleados.duplicated(keep=False)][['empleado_id', 'nombre']].sort_values('empleado_id'))
dfEmpleados = dfEmpleados.drop_duplicates()
print('Estructura o Shape final: ', dfEmpleados.shape)

---
## 4 - Corrección de tipos

`salario` se convierte a `float` eliminando el símbolo `€`. `fecha_ingreso` se convierte a `datetime` desde el formato `dd-mm-yyyy`.

In [ ]:
print("Tipos actuales en 'Empleados': ")
print(dfEmpleados.dtypes)

# .str activa el modo texto para poder usar .replace() sobre cada valor de la columna
# sin .str, .replace() actúa sobre el DataFrame completo y no elimina el símbolo
dfEmpleados['salario'] = dfEmpleados['salario'].str.replace('\u20ac', '', regex=False).astype(float)
dfEmpleados['fecha_ingreso'] = pd.to_datetime(dfEmpleados['fecha_ingreso'], format='%d-%m-%Y')

print('Tipos corregidos:')
print(dfEmpleados[['fecha_ingreso', 'salario']].dtypes)
print(dfEmpleados[['fecha_ingreso', 'salario']].head(5))

---
## 5 - INNER merge

Se une `proyectos` con `empleados` por `empleado_id` para cruzar la información de ambas tablas.

In [ ]:
# pd.merge() une dos DataFrames por una columna que existe en ambos
# dfProyectos → DataFrame izquierdo (el principal: queremos ver todos los proyectos)
# dfEmpleados → DataFrame derecho (del que tomamos la información del empleado asignado)
# on='empleado_id' → columna puente que existe con ese nombre en los dos DataFrames
# how='inner' → devuelve solo los proyectos cuyo empleado_id existe en dfEmpleados
# los proyectos con empleado_id que no exista en dfEmpleados desaparecen del resultado
proyectos_detalle = pd.merge(dfProyectos, dfEmpleados, on='empleado_id', how='inner')

print('Proyectos originales: ', len(dfProyectos))
print('Resultado del merge:  ', len(proyectos_detalle))
print('Primeras filas:')
# seleccionamos solo esas columnas para que la salida sea legible
print(proyectos_detalle[['proyecto_id', 'empleado_id', 'nombre', 'departamento', 'presupuesto']].head())

---
## 6 - Inner vs left merge

Se comparan ambos tipos de merge para identificar qué proyectos no tienen empleado registrado en el sistema.

In [ ]:
# how='inner' → descarta los proyectos cuyo empleado_id no existe en dfEmpleados
merge_inner = pd.merge(dfProyectos, dfEmpleados, on='empleado_id', how='inner')

# how='left' → conserva TODOS los proyectos aunque su empleado_id no exista en dfEmpleados
# donde no hay coincidencia, las columnas de dfEmpleados quedan como NaN
merge_left  = pd.merge(dfProyectos, dfEmpleados, on='empleado_id', how='left')

print('Inner:', len(merge_inner), 'filas')
print('Left: ', len(merge_left),  'filas')
# la diferencia son los proyectos que el inner descartó por no encontrar empleado
print('Proyectos sin empleado registrado:', len(merge_left) - len(merge_inner))

# en el left merge, los proyectos sin coincidencia tienen NaN en 'nombre' (columna de empleados)
# filtramos esas filas para identificar qué proyectos no tienen empleado asignado
huerfanos = merge_left[merge_left['nombre'].isnull()][['proyecto_id', 'empleado_id']]
print('Proyectos sin empleado:')
print(huerfanos)

---
## 7 - Verificación de integridad post-merge

Se verifica que el merge no introdujo filas duplicadas ni perdió proyectos inesperadamente.

In [ ]:
print('Proyectos antes del merge: ', len(dfProyectos))
print('Proyectos después (inner): ', len(proyectos_detalle))

# restamos filas del merge menos filas originales para saber si se ganó o perdió algo
# diff > 0 → el merge añadió filas (hay empleado_ids repetidos en dfEmpleados)
# diff < 0 → el merge perdió filas (proyectos cuyo empleado no existe en dfEmpleados)
# diff = 0 → relación 1:1 perfecta
diff = len(proyectos_detalle) - len(dfProyectos)
if diff > 0:
    # un empleado_id duplicado en dfEmpleados multiplicaría los proyectos que lo referencian
    print('ALERTA: el merge añadió', diff, 'filas → hay duplicados en empleados')
elif diff < 0:
    # abs() convierte el número negativo en positivo para que el mensaje sea legible
    print('INFO: se perdieron', abs(diff), 'proyectos → no tienen empleado registrado')
else:
    print('OK: merge 1:1, sin pérdidas ni duplicaciones')

# verificación final: proyecto_id debe seguir siendo único en el resultado
# si .duplicated().sum() devuelve 0, cada proyecto aparece solo una vez → merge correcto
duplicados_id = proyectos_detalle['proyecto_id'].duplicated().sum()
print('proyecto_ids duplicados post-merge:', duplicados_id)